# Prosperity 4 — Round 0 Market Microstructure Analysis

**Goal:** Understand TOMATOES and EMERALDS market dynamics, detect bot patterns, compare fair value signals, and identify strategy improvement opportunities.

**Data:** 2 days × 10,000 ticks each + ~600 market trades per day

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

DATA_DIR = Path('../prosperity4bt/resources/round0')

# Load prices
prices_d1 = pd.read_csv(DATA_DIR / 'prices_round_0_day_-1.csv', sep=';')
prices_d2 = pd.read_csv(DATA_DIR / 'prices_round_0_day_-2.csv', sep=';')
prices = pd.concat([prices_d2, prices_d1], ignore_index=True)

# Load trades
trades_d1 = pd.read_csv(DATA_DIR / 'trades_round_0_day_-1.csv', sep=';')
trades_d1['day'] = -1
trades_d2 = pd.read_csv(DATA_DIR / 'trades_round_0_day_-2.csv', sep=';')
trades_d2['day'] = -2
trades = pd.concat([trades_d2, trades_d1], ignore_index=True)

print(f'Prices: {len(prices)} rows ({len(prices_d1)} day -1, {len(prices_d2)} day -2)')
print(f'Trades: {len(trades)} rows ({len(trades_d1)} day -1, {len(trades_d2)} day -2)')
print(f'Products: {prices["product"].unique()}')
prices.head()

## Section 1: Price Dynamics

In [ ]:
# Split by product
tom = prices[prices['product'] == 'TOMATOES'].copy()
em = prices[prices['product'] == 'EMERALDS'].copy()

fig, axes = plt.subplots(2, 2, figsize=(16, 8))

for i, (day_val, label) in enumerate([(-2, 'Day -2'), (-1, 'Day -1')]):
    t = tom[tom.day == day_val]
    e = em[em.day == day_val]
    
    axes[0, i].plot(t.timestamp, t.mid_price, linewidth=0.5, color='tomato')
    axes[0, i].set_title(f'TOMATOES {label}')
    axes[0, i].set_xlabel('Timestamp')
    axes[0, i].set_ylabel('Mid Price')
    
    axes[1, i].plot(e.timestamp, e.mid_price, linewidth=0.5, color='green')
    axes[1, i].set_title(f'EMERALDS {label}')
    axes[1, i].set_xlabel('Timestamp')
    axes[1, i].set_ylabel('Mid Price')

plt.tight_layout()
plt.show()

In [ ]:
# Autocorrelation analysis for TOMATOES
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for i, day_val in enumerate([-2, -1]):
    t = tom[tom.day == day_val].reset_index(drop=True)
    returns = t.mid_price.diff().dropna()
    
    # Compute autocorrelation for lags 1-20
    lags = range(1, 21)
    acf = [returns.autocorr(lag=l) for l in lags]
    
    axes[i].bar(lags, acf, color='tomato', alpha=0.7)
    axes[i].axhline(0, color='black', linewidth=0.5)
    axes[i].set_title(f'TOMATOES Return Autocorrelation (Day {day_val})')
    axes[i].set_xlabel('Lag')
    axes[i].set_ylabel('Autocorrelation')
    print(f'Day {day_val} — Lag-1 autocorr: {acf[0]:.4f}')

plt.tight_layout()
plt.show()

In [ ]:
# Volatility analysis — rolling stdev of returns
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for i, day_val in enumerate([-2, -1]):
    t = tom[tom.day == day_val].reset_index(drop=True)
    returns = t.mid_price.diff()
    
    for window in [50, 200, 500]:
        rolling_std = returns.rolling(window).std()
        axes[i].plot(t.timestamp, rolling_std, label=f'Window={window}', linewidth=0.8)
    
    axes[i].set_title(f'TOMATOES Rolling Volatility (Day {day_val})')
    axes[i].set_xlabel('Timestamp')
    axes[i].set_ylabel('Rolling Stdev of Returns')
    axes[i].legend()

plt.tight_layout()
plt.show()

# Summary stats
for day_val in [-2, -1]:
    t = tom[tom.day == day_val]
    e = em[em.day == day_val]
    print(f'\nDay {day_val}:')
    print(f'  TOMATOES mid — mean: {t.mid_price.mean():.2f}, std: {t.mid_price.std():.2f}, min: {t.mid_price.min()}, max: {t.mid_price.max()}')
    print(f'  EMERALDS mid — mean: {e.mid_price.mean():.2f}, std: {e.mid_price.std():.2f}, min: {e.mid_price.min()}, max: {e.mid_price.max()}')

## Section 2: Order Book Microstructure

In [ ]:
# Spread analysis
prices['spread'] = prices['ask_price_1'] - prices['bid_price_1']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for i, prod in enumerate(['TOMATOES', 'EMERALDS']):
    p = prices[prices['product'] == prod]
    for day_val in [-2, -1]:
        d = p[p.day == day_val]
        axes[i].hist(d.spread, bins=30, alpha=0.5, label=f'Day {day_val}')
    axes[i].set_title(f'{prod} Spread Distribution')
    axes[i].set_xlabel('Spread (ask1 - bid1)')
    axes[i].set_ylabel('Count')
    axes[i].legend()

plt.tight_layout()
plt.show()

# Spread stats
for prod in ['TOMATOES', 'EMERALDS']:
    p = prices[prices['product'] == prod]
    print(f'{prod} spread — mean: {p.spread.mean():.2f}, median: {p.spread.median():.0f}, '
          f'min: {p.spread.min()}, max: {p.spread.max()}, mode: {p.spread.mode().iloc[0]}')

In [ ]:
# Order book depth analysis — volume at each level
for prod in ['TOMATOES', 'EMERALDS']:
    p = prices[prices['product'] == prod]
    print(f'\n{prod} Order Book Depth:')
    for level in [1, 2, 3]:
        bid_vol = p[f'bid_volume_{level}'].dropna()
        ask_vol = p[f'ask_volume_{level}'].dropna()
        bid_pct = (bid_vol > 0).sum() / len(p) * 100 if len(bid_vol) > 0 else 0
        ask_pct = (ask_vol > 0).sum() / len(p) * 100 if len(ask_vol) > 0 else 0
        print(f'  Level {level}: bid_vol mean={bid_vol.mean():.1f} (present {bid_pct:.0f}%), '
              f'ask_vol mean={ask_vol.mean():.1f} (present {ask_pct:.0f}%)')

In [ ]:
# Detect bot requoting patterns — how often do bid/ask prices change?
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for i, prod in enumerate(['TOMATOES', 'EMERALDS']):
    p = prices[(prices['product'] == prod) & (prices.day == -1)].reset_index(drop=True)
    
    bid_changes = (p.bid_price_1.diff() != 0).astype(int)
    ask_changes = (p.ask_price_1.diff() != 0).astype(int)
    
    # Rolling frequency of changes per 100 ticks
    bid_freq = bid_changes.rolling(100).sum()
    ask_freq = ask_changes.rolling(100).sum()
    
    axes[i].plot(p.timestamp, bid_freq, label='Bid changes/100 ticks', alpha=0.7)
    axes[i].plot(p.timestamp, ask_freq, label='Ask changes/100 ticks', alpha=0.7)
    axes[i].set_title(f'{prod} Quote Update Frequency (Day -1)')
    axes[i].set_xlabel('Timestamp')
    axes[i].legend()
    
    # Summary
    total_changes = bid_changes.sum() + ask_changes.sum()
    print(f'{prod}: {total_changes} quote changes in {len(p)} ticks ({total_changes/len(p)*100:.1f}%)')

plt.tight_layout()
plt.show()

In [ ]:
# Persistent market maker detection
# Check: how often is the bid/ask at the same price level for extended periods?
for prod in ['TOMATOES', 'EMERALDS']:
    p = prices[(prices['product'] == prod) & (prices.day == -1)].reset_index(drop=True)
    
    # Count consecutive ticks at same bid_price_1
    bid_runs = (p.bid_price_1 != p.bid_price_1.shift()).cumsum()
    run_lengths = bid_runs.groupby(bid_runs).transform('count')
    
    print(f'\n{prod} bid_price_1 persistence (Day -1):')
    print(f'  Mean run length: {run_lengths.mean():.1f} ticks')
    print(f'  Max run length: {run_lengths.max()} ticks')
    print(f'  Median run length: {run_lengths.median():.0f} ticks')
    
    # Most common bid prices
    print(f'  Top 5 bid prices: {p.bid_price_1.value_counts().head().to_dict()}')
    print(f'  Top 5 ask prices: {p.ask_price_1.value_counts().head().to_dict()}')

## Section 3: Bot Detection (The Real Alpha)

In [ ]:
# Market trade analysis
print('Trade summary:')
print(trades.groupby(['day', 'symbol']).agg(
    count=('quantity', 'count'),
    total_volume=('quantity', 'sum'),
    avg_qty=('quantity', 'mean'),
    min_price=('price', 'min'),
    max_price=('price', 'max'),
    mean_price=('price', 'mean')
).round(2))

print('\nTrade quantity distribution:')
print(trades.groupby('symbol')['quantity'].describe().round(2))

In [ ]:
# Olivia detection: look for large trades (10-15 lots) at daily min/max prices
fig, axes = plt.subplots(2, 2, figsize=(16, 8))

for col, day_val in enumerate([-2, -1]):
    for row, symbol in enumerate(['TOMATOES', 'EMERALDS']):
        ax = axes[row, col]
        
        # Get mid-prices for this day/product
        p = prices[(prices['product'] == symbol) & (prices.day == day_val)]
        t = trades[(trades.symbol == symbol) & (trades.day == day_val)]
        
        ax.plot(p.timestamp, p.mid_price, linewidth=0.5, color='gray', alpha=0.5, label='Mid price')
        
        # Scatter trades — size proportional to quantity, color by quantity
        if len(t) > 0:
            scatter = ax.scatter(t.timestamp, t.price, c=t.quantity, s=t.quantity*15, 
                               cmap='YlOrRd', alpha=0.7, edgecolors='black', linewidth=0.5)
            plt.colorbar(scatter, ax=ax, label='Quantity')
            
            # Mark large trades (potential Olivia)
            large = t[t.quantity >= 8]
            if len(large) > 0:
                ax.scatter(large.timestamp, large.price, s=200, facecolors='none', 
                          edgecolors='red', linewidth=2, label=f'Large trades (qty>=8): {len(large)}')
        
        ax.set_title(f'{symbol} Day {day_val}')
        ax.legend(fontsize=8)

plt.suptitle('Market Trades — Size = Volume, Color = Quantity\nRed circles = potential Olivia-like large trades', y=1.02)
plt.tight_layout()
plt.show()

# Print large trade details
print('\nLarge trades (qty >= 8):')
large_trades = trades[trades.quantity >= 8].sort_values(['day', 'symbol', 'timestamp'])
if len(large_trades) > 0:
    print(large_trades.to_string(index=False))
else:
    print('  None found (max qty might be smaller in tutorial round)')

In [ ]:
# Trade direction inference and cumulative flow
# Infer direction: compare trade price to mid-price at that timestamp
trade_analysis = []

for _, trade in trades.iterrows():
    p = prices[(prices['product'] == trade.symbol) & 
              (prices.day == trade.day) & 
              (prices.timestamp == trade.timestamp)]
    if len(p) == 0:
        # Find closest timestamp
        p_all = prices[(prices['product'] == trade.symbol) & (prices.day == trade.day)]
        idx = (p_all.timestamp - trade.timestamp).abs().idxmin()
        p = p_all.loc[[idx]]
    
    mid = p.mid_price.iloc[0]
    bid1 = p.bid_price_1.iloc[0]
    ask1 = p.ask_price_1.iloc[0]
    
    # Classify: trade at/below bid = sell aggressor, at/above ask = buy aggressor
    if trade.price <= bid1:
        direction = -1  # sell aggressor
    elif trade.price >= ask1:
        direction = 1   # buy aggressor
    else:
        direction = 1 if trade.price > mid else -1  # inside spread
    
    trade_analysis.append({
        **trade.to_dict(),
        'mid': mid, 'bid1': bid1, 'ask1': ask1,
        'direction': direction,
        'signed_qty': direction * trade.quantity
    })

ta = pd.DataFrame(trade_analysis)

# Cumulative flow
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for col, day_val in enumerate([-2, -1]):
    for row, symbol in enumerate(['TOMATOES', 'EMERALDS']):
        subset = ta[(ta.symbol == symbol) & (ta.day == day_val)].sort_values('timestamp')
        cum_flow = subset.signed_qty.cumsum()
        
        axes[row, col].plot(subset.timestamp, cum_flow, color='blue', linewidth=1)
        axes[row, col].axhline(0, color='black', linewidth=0.5, linestyle='--')
        axes[row, col].set_title(f'{symbol} Cumulative Trade Flow (Day {day_val})')
        axes[row, col].set_xlabel('Timestamp')
        axes[row, col].set_ylabel('Cumulative Signed Quantity')
        
        # Stats
        buys = subset[subset.direction == 1]
        sells = subset[subset.direction == -1]
        print(f'{symbol} Day {day_val}: {len(buys)} buy trades ({buys.quantity.sum()} vol), '
              f'{len(sells)} sell trades ({sells.quantity.sum()} vol), '
              f'net flow: {subset.signed_qty.sum()}')

plt.tight_layout()
plt.show()

In [ ]:
# Trade clustering: when do trades happen? Is there a pattern?
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for i, symbol in enumerate(['TOMATOES', 'EMERALDS']):
    t = trades[trades.symbol == symbol]
    
    # Inter-trade time
    for day_val in [-2, -1]:
        td = t[t.day == day_val].sort_values('timestamp')
        inter = td.timestamp.diff().dropna()
        axes[i].hist(inter, bins=50, alpha=0.5, label=f'Day {day_val} (mean={inter.mean():.0f}ms)')
    
    axes[i].set_title(f'{symbol} Inter-Trade Time')
    axes[i].set_xlabel('Milliseconds between trades')
    axes[i].set_ylabel('Count')
    axes[i].legend()

plt.tight_layout()
plt.show()

## Section 4: Fair Value Signal Comparison

In [ ]:
# Compute fair value signals for TOMATOES
def compute_signals(product_prices):
    df = product_prices.copy().reset_index(drop=True)
    
    # 1. Simple mid
    df['simple_mid'] = (df.bid_price_1 + df.ask_price_1) / 2
    
    # 2. Microprice
    bid_vol = df.bid_volume_1.fillna(0)
    ask_vol = df.ask_volume_1.fillna(0)
    total_vol = bid_vol + ask_vol
    imb = np.where(total_vol > 0, bid_vol / total_vol, 0.5)
    df['microprice'] = df.bid_price_1 + imb * (df.ask_price_1 - df.bid_price_1)
    
    # 3. VWAP across levels
    bid_vwap_num = (df.bid_price_1.fillna(0) * df.bid_volume_1.fillna(0) + 
                    df.bid_price_2.fillna(0) * df.bid_volume_2.fillna(0) +
                    df.bid_price_3.fillna(0) * df.bid_volume_3.fillna(0))
    bid_vwap_den = (df.bid_volume_1.fillna(0) + df.bid_volume_2.fillna(0) + df.bid_volume_3.fillna(0))
    ask_vwap_num = (df.ask_price_1.fillna(0) * df.ask_volume_1.fillna(0) + 
                    df.ask_price_2.fillna(0) * df.ask_volume_2.fillna(0) +
                    df.ask_price_3.fillna(0) * df.ask_volume_3.fillna(0))
    ask_vwap_den = (df.ask_volume_1.fillna(0) + df.ask_volume_2.fillna(0) + df.ask_volume_3.fillna(0))
    
    bid_vwap = np.where(bid_vwap_den > 0, bid_vwap_num / bid_vwap_den, df.bid_price_1)
    ask_vwap = np.where(ask_vwap_den > 0, ask_vwap_num / ask_vwap_den, df.ask_price_1)
    df['vwap_mid'] = (bid_vwap + ask_vwap) / 2
    
    # 4. Regression on last 4 microprices (trader-4 approach)
    coef = [0.059694, 0.117270, 0.244154, 0.578440]
    intercept = 2.208667
    df['regression'] = intercept
    for i, c in enumerate(coef):
        df['regression'] += c * df['microprice'].shift(4 - i)
    
    # 5. EMA of mid
    df['ema_mid'] = df['simple_mid'].ewm(span=10).mean()
    
    return df

# Compute for TOMATOES
tom_signals = {}
for day_val in [-2, -1]:
    t = prices[(prices['product'] == 'TOMATOES') & (prices.day == day_val)]
    tom_signals[day_val] = compute_signals(t)

print('Signals computed. Columns:', list(tom_signals[-1].columns[-5:]))

In [ ]:
# Compare signals — which best predicts the NEXT mid-price?
signal_names = ['simple_mid', 'microprice', 'vwap_mid', 'regression', 'ema_mid']

results = []
for day_val in [-2, -1]:
    df = tom_signals[day_val].copy()
    target = df.simple_mid.shift(-1)  # next tick's mid
    
    for signal in signal_names:
        valid = df[signal].notna() & target.notna()
        if valid.sum() < 100:
            continue
        error = (df.loc[valid, signal] - target[valid])
        results.append({
            'day': day_val,
            'signal': signal,
            'MAE': error.abs().mean(),
            'RMSE': np.sqrt((error**2).mean()),
            'bias': error.mean(),
            'n': valid.sum()
        })

results_df = pd.DataFrame(results)
print('\nSignal comparison — predicting next TOMATOES mid-price:')
print(results_df.pivot(index='signal', columns='day', values='RMSE').round(4).to_string())
print('\n(Lower RMSE = better prediction)')

In [ ]:
# Compare signals against actual TRADE prices (more meaningful than next mid)
trade_results = []

for day_val in [-2, -1]:
    df = tom_signals[day_val]
    day_trades = trades[(trades.symbol == 'TOMATOES') & (trades.day == day_val)]
    
    for _, trade in day_trades.iterrows():
        # Find the signal values at or just before this trade's timestamp
        mask = df.timestamp <= trade.timestamp
        if mask.sum() == 0:
            continue
        row = df[mask].iloc[-1]
        
        for signal in signal_names:
            if pd.notna(row[signal]):
                trade_results.append({
                    'day': day_val,
                    'signal': signal,
                    'error': row[signal] - trade.price,
                    'abs_error': abs(row[signal] - trade.price)
                })

tr_df = pd.DataFrame(trade_results)
print('Signal comparison — predicting actual TRADE prices:')
summary = tr_df.groupby(['signal', 'day']).agg(
    MAE=('abs_error', 'mean'),
    RMSE=('error', lambda x: np.sqrt((x**2).mean())),
    bias=('error', 'mean'),
    n=('error', 'count')
).round(4)
print(summary.to_string())
print('\n(Lower MAE/RMSE = better signal for fair value)')

## Section 5: Position Limit Analysis

How often does the market-making strategy hit the ±80 limit? How much PnL is lost from being stuck at limits?

In [ ]:
# Simulate simplified position tracking for a market-making strategy
# We can't run the actual trader here, but we can estimate position behavior
# by looking at when the strategy would buy/sell based on order book prices

LIMIT = 80

def simulate_mm_position(product_prices, product_trades, fair_value_fn, day_val):
    """Simulate a basic market-making strategy's position over time."""
    p = product_prices[product_prices.day == day_val].reset_index(drop=True)
    t = product_trades[product_trades.day == day_val].set_index('timestamp')
    
    position = 0
    positions = []
    at_limit_count = 0
    
    for _, row in p.iterrows():
        fair = fair_value_fn(row)
        
        # Simulate taking: buy asks below fair, sell bids above fair
        if row.ask_price_1 <= fair and position < LIMIT:
            qty = min(abs(int(row.ask_volume_1)), LIMIT - position)
            position += qty
        if row.bid_price_1 >= fair and position > -LIMIT:
            qty = min(int(row.bid_volume_1), LIMIT + position)
            position -= qty
        
        at_limit = abs(position) >= LIMIT
        at_limit_count += at_limit
        positions.append({'timestamp': row.timestamp, 'position': position, 'at_limit': at_limit})
    
    return pd.DataFrame(positions), at_limit_count

# EMERALDS: fair = 10000
em_pos, em_at_limit = simulate_mm_position(
    em, trades[trades.symbol == 'EMERALDS'], 
    lambda row: 10000, -1
)

# TOMATOES: fair = mid_price
tom_pos, tom_at_limit = simulate_mm_position(
    tom, trades[trades.symbol == 'TOMATOES'],
    lambda row: row.mid_price, -1
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(tom_pos.timestamp, tom_pos.position, linewidth=0.5, color='tomato')
axes[0].axhline(LIMIT, color='red', linestyle='--', linewidth=0.5)
axes[0].axhline(-LIMIT, color='red', linestyle='--', linewidth=0.5)
axes[0].axhline(0, color='black', linestyle='-', linewidth=0.5)
axes[0].set_title(f'TOMATOES Position (Day -1)\nAt limit: {tom_at_limit}/{len(tom_pos)} ticks ({tom_at_limit/len(tom_pos)*100:.1f}%)')
axes[0].set_ylabel('Position')

axes[1].plot(em_pos.timestamp, em_pos.position, linewidth=0.5, color='green')
axes[1].axhline(LIMIT, color='red', linestyle='--', linewidth=0.5)
axes[1].axhline(-LIMIT, color='red', linestyle='--', linewidth=0.5)
axes[1].axhline(0, color='black', linestyle='-', linewidth=0.5)
axes[1].set_title(f'EMERALDS Position (Day -1)\nAt limit: {em_at_limit}/{len(em_pos)} ticks ({em_at_limit/len(em_pos)*100:.1f}%)')
axes[1].set_ylabel('Position')

plt.tight_layout()
plt.show()

## Section 6: Strategy Comparison

Compare the 5 trader variants on key metrics beyond just PnL.

In [ ]:
# PnL summary from backtester runs (recorded earlier)
strategies = {
    'trader.py': {'day_-2': 6382, 'day_-1': 5834, 'total': 12216, 'desc': 'EMA fair + skew'},
    'trader-1.py': {'day_-2': 15299, 'day_-1': 14456, 'total': 29755, 'desc': 'Regression mid + undercut'},
    'trader-2.py': {'day_-2': 15424, 'day_-1': 14359, 'total': 29784, 'desc': 'jmerle base + mid fair'},
    'trader-3.py': {'day_-2': 15266, 'day_-1': 13790, 'total': 29056, 'desc': 'Imbalance k + regression'},
    'trader-4.py': {'day_-2': 15703, 'day_-1': 13976, 'total': 29679, 'desc': 'Microprice regression'},
}

strat_df = pd.DataFrame(strategies).T
strat_df['day_consistency'] = strat_df['day_-1'] / strat_df['day_-2']  # closer to 1 = more consistent
strat_df['est_website_2k'] = (strat_df['total'] * (2000 / 20000)).astype(int)

print('Strategy Comparison:')
print(strat_df[['desc', 'day_-2', 'day_-1', 'total', 'day_consistency', 'est_website_2k']].to_string())
print('\nday_consistency = day-1 PnL / day-2 PnL (closer to 1.0 = less overfit)')
print('est_website_2k = estimated 2K tick website PnL')

In [ ]:
# Visualize strategy comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = list(strategies.keys())
x = range(len(names))

# PnL by day
d2 = [strategies[n]['day_-2'] for n in names]
d1 = [strategies[n]['day_-1'] for n in names]
axes[0].bar([i - 0.15 for i in x], d2, 0.3, label='Day -2', color='steelblue')
axes[0].bar([i + 0.15 for i in x], d1, 0.3, label='Day -1', color='coral')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(names, rotation=30, ha='right')
axes[0].set_title('PnL by Day')
axes[0].set_ylabel('PnL')
axes[0].legend()

# Consistency
consistency = [strategies[n]['day_-1'] / strategies[n]['day_-2'] for n in names]
colors = ['green' if 0.85 <= c <= 1.15 else 'orange' if 0.7 <= c <= 1.3 else 'red' for c in consistency]
axes[1].bar(x, consistency, color=colors)
axes[1].axhline(1.0, color='black', linestyle='--', linewidth=0.5)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(names, rotation=30, ha='right')
axes[1].set_title('Day-to-Day Consistency (Day-1 / Day-2 PnL)')
axes[1].set_ylabel('Ratio')

plt.tight_layout()
plt.show()

## Key Takeaways

Run all cells above and summarize findings:
1. **Autocorrelation**: Is it actually -0.43? This drives mean-reversion strategy tuning.
2. **Best fair value signal**: Which has lowest RMSE vs trade prices?
3. **Bot patterns**: Any Olivia-like large trades at extremes?
4. **Position management**: How much time is spent at limits?
5. **Day consistency**: Which strategy generalizes best across days?
6. **Spread dynamics**: When is the spread tightest? That's when to be most aggressive.